In [2]:
from google.colab import files
from pathlib import Path
import hashlib
import zipfile
import json
import stat

print("=" * 90)
print("R5-11A - RESTORE AND VERIFY CHECKPOINT")
print("=" * 90)

EXPECTED_ZIP_SHA = "86e42747b943bb33abe711e22a073c6fd46183f4dbf917ba35018c5f316a7d6b"
EXPECTED_POLICY_SHA = "b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243"

EXPECTED_RESULTS = {
    "R5_FROZEN_heldout_decisions.csv": "b3d1328e5d1926fe3a682542b342d51b4300d24d50a1525a0b25eab43292e0f6",
    "R5_FROZEN_system_comparison.csv": "d2cb6a157cca63c96c6b6bfbfedbf050890c7b3486bca049f42e74eafb82bfe2",
    "R5_FROZEN_R4_residual_error_verification.csv": "939c46ff66229a625a1ee6b98dbfa8412f471b83ac91832605d408b49ac365c9",
    "R5_FROZEN_results_by_uc_kind.csv": "c35a55f5580047ddc00af93a5ac76af1dab7f173da85981ee61ffe172a778dc9",
    "R5_10_FROZEN_HELDOUT_SUMMARY.json": "60d8f4411f3009d358e4ccddf0b0925d315e2684cb83b11b6c868406d3446a3e"
}

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block

SyntaxError: invalid syntax (1659682014.py, line 26)

In [3]:

from google.colab import files
from pathlib import Path
import hashlib
import zipfile
import json

print("R5-11A: RESTORE AND VERIFY CHECKPOINT")

expected_zip = "86e42747b943bb33abe711e22a073c6fd46183f4dbf917ba35018c5f316a7d6b"
expected_policy = "b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243"

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

matches = list(Path("/content").glob("XAgentic_R5_CHECKPOINT*.zip"))
matches = [p for p in matches if sha256(p) == expected_zip]

if not matches:
    print("Please upload your R5 checkpoint ZIP")
    uploaded = files.upload()
    matches = [Path("/content") / name for name in uploaded]
    matches = [p for p in matches if sha256(p) == expected_zip]

assert len(matches) == 1, "STOP: checkpoint ZIP verification failed"
checkpoint = matches[0]

print("PASS: ZIP SHA256 verified")
print("ZIP size:", checkpoint.stat().st_size)

root = Path("/content/R5_work")

with zipfile.ZipFile(checkpoint, "r") as z:
    assert z.testzip() is None, "STOP: ZIP CRC failed"
    for member in z.namelist():
        p = Path(member)
        assert not p.is_absolute() and ".." not in p.parts
        assert p.parts[0] == "R5_work"
    if not root.exists():
        z.extractall("/content")

print("PASS: workspace restored")

policy_file = root / "R5_results/R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json"
assert policy_file.exists(), "STOP: frozen policy missing"
assert sha256(policy_file) == expected_policy, "STOP: policy hash mismatch"

with open(policy_file) as f:
    policy = json.load(f)

threshold = policy["semantic_verifier"]["frozen_semantic_support_threshold"]
assert float(threshold) == -0.0275677986595984

print("PASS: frozen policy verified")
print("R5 threshold:", threshold)

result_file = root / "R5_results/R5_10_FROZEN_HELDOUT_SUMMARY.json"
assert result_file.exists(), "STOP: held-out summary missing"

with open(result_file) as f:
    result = json.load(f)

assert result["total"] == 360
assert result["r4"]["autonomous_errors"] == 5
assert result["r5"]["r4_errors_captured"] == 3
assert result["r5"]["autonomous_errors"] == 2

state_file = root / "audit/R5_RESTART_STATE.json"
assert state_file.exists(), "STOP: restart state missing"

with open(state_file) as f:
    state = json.load(f)

assert state["completed_through"] == "R5-10B"
assert state["next_stage"] == "R5-11"

print("PASS: frozen held-out results verified")
print("R4 errors:", result["r4"]["autonomous_errors"])
print("R5 captured:", result["r5"]["r4_errors_captured"])
print("R5 remaining:", result["r5"]["autonomous_errors"])
print("R5 coverage:", result["r5"]["coverage"])
print("R5 selective risk:", result["r5"]["selective_risk"])

print("=" * 70)
print("PASS - R5 CHECKPOINT RESTORED SUCCESSFULLY")
print("READY FOR R5-11")
print("=" * 70)


R5-11A: RESTORE AND VERIFY CHECKPOINT
Please upload your R5 checkpoint ZIP


Saving XAgentic_R5_CHECKPOINT_AFTER_R5_10B_2026-10-07.zip to XAgentic_R5_CHECKPOINT_AFTER_R5_10B_2026-10-07.zip
PASS: ZIP SHA256 verified
ZIP size: 968467
PASS: workspace restored
PASS: frozen policy verified
R5 threshold: -0.0275677986595984
PASS: frozen held-out results verified
R4 errors: 5
R5 captured: 3
R5 remaining: 2
R5 coverage: 0.7777777777777778
R5 selective risk: 0.007142857142857143
PASS - R5 CHECKPOINT RESTORED SUCCESSFULLY
READY FOR R5-11


In [4]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

root = Path("/content/R5_work")
results = root / "R5_results"
audit = root / "audit"

print("=" * 85)
print("R5-11B - FROZEN R3 / R4 / R5 COMPARISON")
print("=" * 85)

policy_file = results / "R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json"
expected_sha = "b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243"
actual_sha = hashlib.sha256(policy_file.read_bytes()).hexdigest()
assert actual_sha == expected_sha, "STOP: frozen policy changed"

decisions = pd.read_csv(results / "R5_FROZEN_heldout_decisions.csv")
assert len(decisions) == 360

def flag(col):
    x = decisions[col]
    if pd.api.types.is_bool_dtype(x):
        return x.fillna(False).astype(bool)
    return x.astype(str).str.lower().isin(["true", "1"])

systems = [
    ("Frozen R3", "r4_r3_autonomous", "r4_r3_wrong"),
    ("Frozen R4", "r4_final_autonomous", "r4_final_wrong"),
    ("Frozen R4 + R5", "r5_final_autonomous", "r5_final_wrong")
]

rows = []
for name, auto_col, wrong_col in systems:
    assert auto_col in decisions.columns, "Missing: " + auto_col
    assert wrong_col in decisions.columns, "Missing: " + wrong_col
    auto = flag(auto_col)
    wrong = flag(wrong_col)
    assert not (wrong & ~auto).any()
    n_auto = int(auto.sum())
    n_wrong = int(wrong.sum())
    n_correct = n_auto - n_wrong
    rows.append({
        "system": name,
        "total": 360,
        "autonomous": n_auto,
        "escalated": 360 - n_auto,
        "autonomous_correct": n_correct,
        "autonomous_errors": n_wrong,
        "coverage": n_auto / 360,
        "autonomous_accuracy": n_correct / n_auto,
        "selective_risk": n_wrong / n_auto,
        "wrong_autonomous_over_total": n_wrong / 360
    })

comparison = pd.DataFrame(rows)

assert comparison["autonomous"].tolist() == [341, 306, 280]
assert comparison["autonomous_errors"].tolist() == [12, 5, 2]
assert comparison["autonomous_correct"].tolist() == [329, 301, 278]

r3, r4, r5 = rows

analysis = {
    "r3_to_r4_errors_removed": r3["autonomous_errors"] - r4["autonomous_errors"],
    "r4_to_r5_errors_removed": r4["autonomous_errors"] - r5["autonomous_errors"],
    "r3_to_r5_errors_removed": r3["autonomous_errors"] - r5["autonomous_errors"],
    "r4_to_r5_relative_selective_risk_reduction": (
        1 - r5["selective_risk"] / r4["selective_risk"]
    ),
    "r3_to_r5_relative_selective_risk_reduction": (
        1 - r5["selective_risk"] / r3["selective_risk"]
    ),
    "r4_to_r5_coverage_change_percentage_points": (
        r5["coverage"] - r4["coverage"]
    ) * 100,
    "r4_to_r5_accuracy_change_percentage_points": (
        r5["autonomous_accuracy"] - r4["autonomous_accuracy"]
    ) * 100,
    "frozen_policy_sha256": actual_sha,
    "heldout_retuning": False
}

print("\nSYSTEM COMPARISON")
print("-" * 85)
print(comparison.round(6).to_string(index=False))

print("\nRISK AND COVERAGE ANALYSIS")
print("-" * 85)
for key, value in analysis.items():
    print(f"{key}: {value}")

comparison_path = results / "R5_11B_three_system_comparison.csv"
analysis_path = results / "R5_11B_risk_coverage_analysis.json"

comparison.to_csv(comparison_path, index=False)

with open(analysis_path, "w", encoding="utf-8") as f:
    json.dump(analysis, f, indent=2)

print("\nEVIDENCE FILES")
for p in [comparison_path, analysis_path]:
    print(p.name, hashlib.sha256(p.read_bytes()).hexdigest())

assert hashlib.sha256(policy_file.read_bytes()).hexdigest() == expected_sha

print("\nPASS - R5-11B frozen comparison completed")
print("Policy unchanged. No held-out retuning.")


R5-11B - FROZEN R3 / R4 / R5 COMPARISON

SYSTEM COMPARISON
-------------------------------------------------------------------------------------
        system  total  autonomous  escalated  autonomous_correct  autonomous_errors  coverage  autonomous_accuracy  selective_risk  wrong_autonomous_over_total
     Frozen R3    360         341         19                 329                 12  0.947222             0.964809        0.035191                     0.033333
     Frozen R4    360         306         54                 301                  5  0.850000             0.983660        0.016340                     0.013889
Frozen R4 + R5    360         280         80                 278                  2  0.777778             0.992857        0.007143                     0.005556

RISK AND COVERAGE ANALYSIS
-------------------------------------------------------------------------------------
r3_to_r4_errors_removed: 7
r4_to_r5_errors_removed: 3
r3_to_r5_errors_removed: 10
r4_to_r5_relative_s

In [5]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

root = Path("/content/R5_work")
results = root / "R5_results"

print("=" * 90)
print("R5-11C - FROZEN SEMANTIC VERIFICATION AUDIT")
print("=" * 90)

policy_path = results / "R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json"
expected = "b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243"

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha(policy_path) == expected

with open(policy_path) as f:
    policy = json.load(f)

tau = float(policy["semantic_verifier"]["frozen_semantic_support_threshold"])

df = pd.read_csv(results / "R5_FROZEN_heldout_decisions.csv")

assert len(df) == 360

def boolean(col):
    return df[col].astype(str).str.lower().isin(["true", "1"])

r4_auto = boolean("r4_final_autonomous")
r4_wrong = boolean("r4_final_wrong")
r5_flag = boolean("r5_additional_escalation")
r5_auto = boolean("r5_final_autonomous")

assert int(r4_auto.sum()) == 306
assert int(r4_wrong.sum()) == 5
assert int(r5_flag.sum()) == 26
assert int((r4_wrong & r5_flag).sum()) == 3
assert int(r5_auto.sum()) == 280

print("\nFROZEN POLICY")
print("Threshold:", tau)
print("Policy SHA256:", sha(policy_path))

# 1. Ablation: remove R5 while retaining frozen R4
ablation = pd.DataFrame([
    {
        "configuration": "R4 without R5 verifier",
        "autonomous": 306,
        "errors": 5,
        "correct": 301,
        "coverage": 306 / 360,
        "selective_risk": 5 / 306
    },
    {
        "configuration": "R4 with frozen R5 verifier",
        "autonomous": 280,
        "errors": 2,
        "correct": 278,
        "coverage": 280 / 360,
        "selective_risk": 2 / 280
    }
])

print("\nVERIFIER ABLATION")
print(ablation.round(6).to_string(index=False))

# 2. Breakdown by use-case category
rows = []

for kind, group in df[r4_auto].groupby("uc_kind"):
    flag = group["r5_additional_escalation"].astype(str).str.lower().isin(["true", "1"])
    wrong = group["r4_final_wrong"].astype(str).str.lower().isin(["true", "1"])

    errors = int(wrong.sum())
    caught = int((flag & wrong).sum())
    flagged = int(flag.sum())

    rows.append({
        "uc_kind": kind,
        "r4_autonomous": len(group),
        "r4_errors": errors,
        "r5_flagged": flagged,
        "errors_captured": caught,
        "correct_escalated": flagged - caught,
        "capture_rate": caught / errors if errors else np.nan
    })

by_kind = pd.DataFrame(rows)

print("\nUSE-CASE CATEGORY AUDIT")
print(by_kind.to_string(index=False))

# 3. Frozen threshold distance analysis
subset = df[r4_auto].copy()
subset["distance_from_frozen_threshold"] = (
    subset["semantic_support_margin"] - tau
)

subset["r4_wrong_bool"] = (
    subset["r4_final_wrong"].astype(str).str.lower().isin(["true", "1"])
)

subset["r5_flag_bool"] = (
    subset["r5_additional_escalation"].astype(str).str.lower().isin(["true", "1"])
)

subset["abs_threshold_distance"] = (
    subset["distance_from_frozen_threshold"].abs()
)

near = subset.nsmallest(
    20, "abs_threshold_distance"
)[[
    "id",
    "uc_kind",
    "r4_wrong_bool",
    "r5_flag_bool",
    "semantic_support_margin",
    "distance_from_frozen_threshold"
]]

print("\n20 CASES CLOSEST TO FROZEN THRESHOLD")
print(near.to_string(index=False))

# 4. Error capture audit
errors = subset[subset["r4_wrong_bool"]].copy()

error_audit = errors[[
    "id",
    "uc_kind",
    "truth",
    "final_pred",
    "semantic_pred",
    "semantic_support_margin",
    "distance_from_frozen_threshold",
    "r5_flag_bool"
]]

print("\nALL FIVE R4 RESIDUAL ERRORS")
print(error_audit.to_string(index=False))

# 5. Save audit evidence
ablation_path = results / "R5_11C_verifier_ablation.csv"
kind_path = results / "R5_11C_category_audit.csv"
near_path = results / "R5_11C_frozen_threshold_proximity.csv"
error_path = results / "R5_11C_residual_error_audit.csv"

ablation.to_csv(ablation_path, index=False)
by_kind.to_csv(kind_path, index=False)
near.to_csv(near_path, index=False)
error_audit.to_csv(error_path, index=False)

print("\nEVIDENCE HASHES")
for p in [ablation_path, kind_path, near_path, error_path]:
    print(p.name, sha(p))

assert sha(policy_path) == expected

print("\nPASS - R5-11C frozen verifier audit completed")
print("No threshold change. No held-out retuning.")


R5-11C - FROZEN SEMANTIC VERIFICATION AUDIT

FROZEN POLICY
Threshold: -0.0275677986595984
Policy SHA256: b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243

VERIFIER ABLATION
             configuration  autonomous  errors  correct  coverage  selective_risk
    R4 without R5 verifier         306       5      301  0.850000        0.016340
R4 with frozen R5 verifier         280       2      278  0.777778        0.007143

USE-CASE CATEGORY AUDIT
uc_kind  r4_autonomous  r4_errors  r5_flagged  errors_captured  correct_escalated  capture_rate
generic             88          1          24                1                 23           1.0
  known            123          0           0                0                  0           NaN
 unseen             95          4           2                2                  0           0.5

20 CASES CLOSEST TO FROZEN THRESHOLD
  id uc_kind  r4_wrong_bool  r5_flag_bool  semantic_support_margin  distance_from_frozen_threshold
 605 generic       

In [6]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

print("=" * 85)
print("R5-11D - PAIRED BOOTSTRAP ROBUSTNESS ANALYSIS")
print("=" * 85)

root = Path("/content/R5_work")
out = root / "R5_results"

policy = out / "R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json"
expected = "b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243"

def sha(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

assert sha(policy) == expected

df = pd.read_csv(out / "R5_FROZEN_heldout_decisions.csv")
assert len(df) == 360

def flags(name):
    return df[name].astype(str).str.lower().isin(["true", "1"]).to_numpy()

a4 = flags("r4_final_autonomous")
w4 = flags("r4_final_wrong")
a5 = flags("r5_final_autonomous")
w5 = flags("r5_final_wrong")

assert a4.sum() == 306
assert w4.sum() == 5
assert a5.sum() == 280
assert w5.sum() == 2

assert np.all(a5 <= a4)
assert np.all(w5 <= w4)

n = len(df)
rng = np.random.default_rng(42)
B = 10000

records = []

for _ in range(B):
    idx = rng.integers(0, n, n)

    n4 = int(a4[idx].sum())
    n5 = int(a5[idx].sum())

    if n4 == 0 or n5 == 0:
        continue

    e4 = int(w4[idx].sum())
    e5 = int(w5[idx].sum())

    risk4 = e4 / n4
    risk5 = e5 / n5

    records.append([
        risk4,
        risk5,
        risk4 - risk5,
        n4 / n,
        n5 / n,
        (n4 - n5) / n,
        e4 - e5
    ])

columns = [
    "r4_risk",
    "r5_risk",
    "absolute_risk_reduction",
    "r4_coverage",
    "r5_coverage",
    "coverage_cost",
    "errors_captured"
]

boot = pd.DataFrame(records, columns=columns)

summary = []

for metric in columns:
    x = boot[metric].to_numpy()
    summary.append({
        "metric": metric,
        "bootstrap_mean": float(x.mean()),
        "ci_95_lower": float(np.quantile(x, 0.025)),
        "ci_95_upper": float(np.quantile(x, 0.975))
    })

ci = pd.DataFrame(summary)

print("\nBOOTSTRAP SETTINGS")
print("Held-out cases:", n)
print("Resamples:", B)
print("Seed: 42")
print("Method: paired case-level bootstrap")

print("\n95% PERCENTILE CONFIDENCE INTERVALS")
print(ci.round(6).to_string(index=False))

print("\nOBSERVED FROZEN RESULTS")
print("R4 selective risk:", 5 / 306)
print("R5 selective risk:", 2 / 280)
print("Absolute risk reduction:", 5 / 306 - 2 / 280)
print("Coverage cost:", (306 - 280) / 360)
print("Additional errors captured:", 3)

ci_path = out / "R5_11D_bootstrap_confidence_intervals.csv"
config_path = out / "R5_11D_bootstrap_configuration.json"

ci.to_csv(ci_path, index=False)

config = {
    "method": "paired_case_level_bootstrap",
    "seed": 42,
    "resamples": B,
    "heldout_rows": n,
    "confidence_level": 0.95,
    "policy_sha256": expected,
    "threshold_retuning": False,
    "interpretation": "Descriptive uncertainty analysis of frozen R4/R5 decisions"
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("\nEVIDENCE HASHES")
print(ci_path.name, sha(ci_path))
print(config_path.name, sha(config_path))

assert sha(policy) == expected

print("\nPASS - R5-11D statistical robustness completed")
print("Frozen policy unchanged")


R5-11D - PAIRED BOOTSTRAP ROBUSTNESS ANALYSIS

BOOTSTRAP SETTINGS
Held-out cases: 360
Resamples: 10000
Seed: 42
Method: paired case-level bootstrap

95% PERCENTILE CONFIDENCE INTERVALS
                 metric  bootstrap_mean  ci_95_lower  ci_95_upper
                r4_risk        0.016305     0.003279     0.032258
                r5_risk        0.007133     0.000000     0.018116
absolute_risk_reduction        0.009172    -0.000531     0.021837
            r4_coverage        0.849882     0.811111     0.886111
            r5_coverage        0.777670     0.733333     0.819444
          coverage_cost        0.072211     0.047222     0.100000
        errors_captured        2.991300     0.000000     7.000000

OBSERVED FROZEN RESULTS
R4 selective risk: 0.016339869281045753
R5 selective risk: 0.007142857142857143
Absolute risk reduction: 0.00919701213818861
Coverage cost: 0.07222222222222222
Additional errors captured: 3

EVIDENCE HASHES
R5_11D_bootstrap_confidence_intervals.csv c7c4123225ecb

In [7]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

print("=" * 85)
print("R5-11D - PAIRED BOOTSTRAP ROBUSTNESS ANALYSIS")
print("=" * 85)

root = Path("/content/R5_work")
out = root / "R5_results"

policy = out / "R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json"
expected = "b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243"

def sha(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

assert sha(policy) == expected

df = pd.read_csv(out / "R5_FROZEN_heldout_decisions.csv")
assert len(df) == 360

def flags(name):
    return df[name].astype(str).str.lower().isin(["true", "1"]).to_numpy()

a4 = flags("r4_final_autonomous")
w4 = flags("r4_final_wrong")
a5 = flags("r5_final_autonomous")
w5 = flags("r5_final_wrong")

assert a4.sum() == 306
assert w4.sum() == 5
assert a5.sum() == 280
assert w5.sum() == 2

assert np.all(a5 <= a4)
assert np.all(w5 <= w4)

n = len(df)
rng = np.random.default_rng(42)
B = 10000

records = []

for _ in range(B):
    idx = rng.integers(0, n, n)

    n4 = int(a4[idx].sum())
    n5 = int(a5[idx].sum())

    if n4 == 0 or n5 == 0:
        continue

    e4 = int(w4[idx].sum())
    e5 = int(w5[idx].sum())

    risk4 = e4 / n4
    risk5 = e5 / n5

    records.append([
        risk4,
        risk5,
        risk4 - risk5,
        n4 / n,
        n5 / n,
        (n4 - n5) / n,
        e4 - e5
    ])

columns = [
    "r4_risk",
    "r5_risk",
    "absolute_risk_reduction",
    "r4_coverage",
    "r5_coverage",
    "coverage_cost",
    "errors_captured"
]

boot = pd.DataFrame(records, columns=columns)

summary = []

for metric in columns:
    x = boot[metric].to_numpy()
    summary.append({
        "metric": metric,
        "bootstrap_mean": float(x.mean()),
        "ci_95_lower": float(np.quantile(x, 0.025)),
        "ci_95_upper": float(np.quantile(x, 0.975))
    })

ci = pd.DataFrame(summary)

print("\nBOOTSTRAP SETTINGS")
print("Held-out cases:", n)
print("Resamples:", B)
print("Seed: 42")
print("Method: paired case-level bootstrap")

print("\n95% PERCENTILE CONFIDENCE INTERVALS")
print(ci.round(6).to_string(index=False))

print("\nOBSERVED FROZEN RESULTS")
print("R4 selective risk:", 5 / 306)
print("R5 selective risk:", 2 / 280)
print("Absolute risk reduction:", 5 / 306 - 2 / 280)
print("Coverage cost:", (306 - 280) / 360)
print("Additional errors captured:", 3)

ci_path = out / "R5_11D_bootstrap_confidence_intervals.csv"
config_path = out / "R5_11D_bootstrap_configuration.json"

ci.to_csv(ci_path, index=False)

config = {
    "method": "paired_case_level_bootstrap",
    "seed": 42,
    "resamples": B,
    "heldout_rows": n,
    "confidence_level": 0.95,
    "policy_sha256": expected,
    "threshold_retuning": False,
    "interpretation": "Descriptive uncertainty analysis of frozen R4/R5 decisions"
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("\nEVIDENCE HASHES")
print(ci_path.name, sha(ci_path))
print(config_path.name, sha(config_path))

assert sha(policy) == expected

print("\nPASS - R5-11D statistical robustness completed")
print("Frozen policy unchanged")


R5-11D - PAIRED BOOTSTRAP ROBUSTNESS ANALYSIS

BOOTSTRAP SETTINGS
Held-out cases: 360
Resamples: 10000
Seed: 42
Method: paired case-level bootstrap

95% PERCENTILE CONFIDENCE INTERVALS
                 metric  bootstrap_mean  ci_95_lower  ci_95_upper
                r4_risk        0.016305     0.003279     0.032258
                r5_risk        0.007133     0.000000     0.018116
absolute_risk_reduction        0.009172    -0.000531     0.021837
            r4_coverage        0.849882     0.811111     0.886111
            r5_coverage        0.777670     0.733333     0.819444
          coverage_cost        0.072211     0.047222     0.100000
        errors_captured        2.991300     0.000000     7.000000

OBSERVED FROZEN RESULTS
R4 selective risk: 0.016339869281045753
R5 selective risk: 0.007142857142857143
Absolute risk reduction: 0.00919701213818861
Coverage cost: 0.07222222222222222
Additional errors captured: 3

EVIDENCE HASHES
R5_11D_bootstrap_confidence_intervals.csv c7c4123225ecb

In [8]:

from pathlib import Path
import json
import hashlib

root = Path("/content/R5_work")
out = root / "R5_results"
policy = out / "R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json"

expected = "b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243"

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha(policy) == expected, "STOP: policy changed"

required = [
    "R5_11B_three_system_comparison.csv",
    "R5_11B_risk_coverage_analysis.json",
    "R5_11C_verifier_ablation.csv",
    "R5_11C_category_audit.csv",
    "R5_11C_residual_error_audit.csv",
    "R5_11D_bootstrap_confidence_intervals.csv",
    "R5_11D_bootstrap_configuration.json"
]

for name in required:
    assert (out / name).exists(), "Missing: " + name

report = """# R5 - FINAL FINDINGS AND INTERPRETATION

## 1. Research objective

R5 evaluates a frozen semantic verification layer added after the
R4 selective escalation mechanism in an explainable agentic AI
pipeline for network slicing intent interpretation.

The objective is to identify residual autonomous classification
errors while preserving as many correct autonomous decisions as
possible.

## 2. Experimental protocol

- Development cases: 840
- Held-out evaluation cases: 360
- Held-out policy retuning: prohibited
- Frozen semantic support threshold: -0.0275677986595984
- Frozen policy SHA256: b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243

The semantic verifier was calibrated using development data.
Its policy was frozen before held-out evaluation.

## 3. Frozen system comparison

| Metric | R3 | R4 | R5 |
|---|---:|---:|---:|
| Autonomous decisions | 341 | 306 | 280 |
| Escalated decisions | 19 | 54 | 80 |
| Correct autonomous decisions | 329 | 301 | 278 |
| Autonomous errors | 12 | 5 | 2 |
| Coverage | 94.72% | 85.00% | 77.78% |
| Autonomous accuracy | 96.48% | 98.37% | 99.29% |
| Selective risk | 3.52% | 1.63% | 0.71% |

R5 reduces observed selective risk by 56.29% relative to R4
and 79.70% relative to R3.

The R4-to-R5 coverage cost is 7.22 percentage points.

## 4. Semantic verifier ablation

Without R5:
- Autonomous cases: 306
- Autonomous errors: 5
- Selective risk: 1.634%

With frozen R5:
- Autonomous cases: 280
- Autonomous errors: 2
- Selective risk: 0.714%

R5 additionally escalates 26 cases:
- Previously incorrect decisions captured: 3
- Previously correct decisions escalated: 23
- Residual-error capture rate: 60%

## 5. Use-case category findings

Generic:
- R4 autonomous cases: 88
- R4 errors: 1
- R5 captures: 1
- Correct decisions escalated: 23

Known:
- R4 autonomous cases: 123
- R4 errors: 0
- R5 escalations: 0

Unseen:
- R4 autonomous cases: 95
- R4 errors: 4
- R5 captures: 2
- Correct decisions escalated: 0

All 23 unnecessary R5 escalations occur in generic cases.

## 6. Residual error analysis

Captured errors:
- ID 1197: generic, eMBB predicted mMTC
- ID 240: unseen, eMBB predicted mMTC
- ID 769: unseen, URLLC predicted eMBB

Missed errors:
- ID 658: unseen, URLLC predicted mMTC
- ID 810: unseen, eMBB predicted mMTC

Both missed errors have semantic support margins above
the frozen escalation threshold.

No threshold adjustment was performed.

## 7. Statistical robustness

Method:
- Paired case-level bootstrap
- 10,000 resamples
- Random seed: 42
- Held-out sample size: 360

95% percentile bootstrap confidence intervals:

R4 selective risk:
- Observed: 1.634%
- CI: 0.328% to 3.226%

R5 selective risk:
- Observed: 0.714%
- CI: 0% to 1.812%

Absolute risk reduction:
- Observed: 0.920 percentage points
- CI: -0.053 to 2.184 percentage points

Coverage cost:
- Observed: 7.222 percentage points
- CI: 4.722 to 10.000 percentage points

The absolute risk-reduction confidence interval includes zero.
Consequently, the observed improvement should not be
described as statistically conclusive at the 95% level.

## 8. Novelty and contribution

R5 adds a development-calibrated semantic verification
mechanism to an existing feasibility-aware, selectively
escalated intent interpretation pipeline.

Its experimental contribution is the measured reduction
in residual autonomous errors under a frozen policy,
together with explicit quantification of coverage costs,
category-specific behavior, and statistical uncertainty.

The current experiment does not establish deployment-scale
reliability or universal generalization.

## 9. Limitations

1. Only five residual R4 errors exist in the held-out sample.
2. The confidence interval for risk reduction includes zero.
3. Twenty-three correct generic decisions are escalated.
4. Two unseen-intent errors remain undetected.
5. Evaluation is limited to the frozen benchmark.
6. Runtime deployment, operator feedback, and live
   network-slicing actuation are not established here.

## 10. Future research

Future work may investigate:
- Category-aware semantic verification
- Better treatment of generic intents
- Independent held-out datasets
- Larger residual-error samples
- Runtime overhead and end-to-end latency
- Integration with network slicing orchestration

Any modified policy requires development-only calibration
and evaluation on a fresh untouched test set.

## 11. Conclusion

The frozen R5 verifier reduces observed autonomous errors
from five to two after R4, increasing autonomous accuracy
from 98.366% to 99.286%.

This comes at the cost of reducing coverage from 85.00%
to 77.78%.

The results support the feasibility of semantic verification
as an additional assurance layer, while the uncertainty
analysis and generic-intent false escalations demonstrate
the need for further independent validation.
"""

path = out / "R5_FINAL_FINDINGS_AND_INTERPRETATION.md"
path.write_text(report, encoding="utf-8")

assert path.exists()
assert sha(policy) == expected

print("=" * 80)
print("R5-12A - FINAL FINDINGS DOCUMENT")
print("=" * 80)
print("Written:", path)
print("Bytes:", path.stat().st_size)
print("SHA256:", sha(path))
print("Policy unchanged:", sha(policy) == expected)
print("PASS - R5 final findings document created")


R5-12A - FINAL FINDINGS DOCUMENT
Written: /content/R5_work/R5_results/R5_FINAL_FINDINGS_AND_INTERPRETATION.md
Bytes: 4845
SHA256: f673d78560678da83b8c0a15160163e7d0a01357007da9e387c267276dc2707e
Policy unchanged: True
PASS - R5 final findings document created


In [9]:

from pathlib import Path
from google.colab import files
import hashlib
import json
import zipfile
import csv

print("=" * 80)
print("R5-13A - FINAL EVIDENCE AUDIT AND PACKAGE")
print("=" * 80)

root = Path("/content/R5_work")
results = root / "R5_results"
audit = root / "audit"
audit.mkdir(exist_ok=True)

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1048576), b""):
            h.update(block)
    return h.hexdigest()

policy = results / "R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json"
expected = "b8e5ef13e6895fcfb3656f7257c94727e3cea17d38f08563436d55b6cf0ba243"
assert sha(policy) == expected, "STOP: frozen policy changed"

required = [
    "R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json",
    "R5_FROZEN_heldout_decisions.csv",
    "R5_FROZEN_system_comparison.csv",
    "R5_FROZEN_R4_residual_error_verification.csv",
    "R5_FROZEN_results_by_uc_kind.csv",
    "R5_10_FROZEN_HELDOUT_SUMMARY.json",
    "R5_11B_three_system_comparison.csv",
    "R5_11B_risk_coverage_analysis.json",
    "R5_11C_verifier_ablation.csv",
    "R5_11C_category_audit.csv",
    "R5_11C_frozen_threshold_proximity.csv",
    "R5_11C_residual_error_audit.csv",
    "R5_11D_bootstrap_confidence_intervals.csv",
    "R5_11D_bootstrap_configuration.json",
    "R5_FINAL_FINDINGS_AND_INTERPRETATION.md"
]

for name in required:
    assert (results / name).is_file(), "STOP: missing " + name
    print("PASS:", name)

assert (audit / "R5_09_POLICY_FREEZE_RECORD.json").is_file()
assert (audit / "R5_RESTART_STATE.json").is_file()

assert (root / "R3_recovered").is_dir()
assert (root / "R4_recovered").is_dir()

# Audit every existing file before adding the manifest.
entries = []

for path in sorted(root.rglob("*")):
    if path.is_file() and path.name not in [
        "R5_FINAL_SHA256_MANIFEST.csv",
        "R5_FINAL_PACKAGE_SUMMARY.json"
    ]:
        entries.append({
            "relative_path": path.relative_to(root).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": sha(path)
        })

assert len(entries) >= 40, "STOP: unexpectedly few evidence files"

manifest = audit / "R5_FINAL_SHA256_MANIFEST.csv"

with open(manifest, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(entries)

summary = {
    "experiment": "R5 Frozen Semantic Verification",
    "completed_through": "R5-13A",
    "heldout_cases": 360,
    "development_cases": 840,
    "frozen_policy_sha256": expected,
    "frozen_threshold": -0.0275677986595984,
    "r3_autonomous_errors": 12,
    "r4_autonomous_errors": 5,
    "r5_autonomous_errors": 2,
    "r5_autonomous": 280,
    "r5_coverage": 280 / 360,
    "r5_selective_risk": 2 / 280,
    "heldout_retuning": False,
    "manifest_entries": len(entries),
    "statistical_caution": (
        "The 95% bootstrap CI for absolute risk reduction "
        "includes zero."
    )
}

summary_path = audit / "R5_FINAL_PACKAGE_SUMMARY.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

zip_path = Path("/content/XAgentic_R5_FINAL_EVIDENCE.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for path in sorted(root.rglob("*")):
        if path.is_file():
            z.write(path, arcname=path.relative_to(root.parent))

with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None
    names = z.namelist()
    assert len(names) == len(entries) + 2

assert sha(policy) == expected

print("\nFINAL PACKAGE")
print("ZIP:", zip_path)
print("Size:", zip_path.stat().st_size)
print("Files:", len(names))
print("ZIP SHA256:", sha(zip_path))
print("Manifest entries:", len(entries))
print("Frozen policy unchanged: YES")
print("\nPASS - R5 FINAL PACKAGE VERIFIED")

files.download(str(zip_path))


R5-13A - FINAL EVIDENCE AUDIT AND PACKAGE
PASS: R5_FROZEN_SEMANTIC_VERIFICATION_POLICY.json
PASS: R5_FROZEN_heldout_decisions.csv
PASS: R5_FROZEN_system_comparison.csv
PASS: R5_FROZEN_R4_residual_error_verification.csv
PASS: R5_FROZEN_results_by_uc_kind.csv
PASS: R5_10_FROZEN_HELDOUT_SUMMARY.json
PASS: R5_11B_three_system_comparison.csv
PASS: R5_11B_risk_coverage_analysis.json
PASS: R5_11C_verifier_ablation.csv
PASS: R5_11C_category_audit.csv
PASS: R5_11C_frozen_threshold_proximity.csv
PASS: R5_11C_residual_error_audit.csv
PASS: R5_11D_bootstrap_confidence_intervals.csv
PASS: R5_11D_bootstrap_configuration.json
PASS: R5_FINAL_FINDINGS_AND_INTERPRETATION.md

FINAL PACKAGE
ZIP: /content/XAgentic_R5_FINAL_EVIDENCE.zip
Size: 980342
Files: 93
ZIP SHA256: 29349a6717f9a89dcfdb957c6c91d8df640de1cb57a1ffa7f91e252dfc24cdb6
Manifest entries: 91
Frozen policy unchanged: YES

PASS - R5 FINAL PACKAGE VERIFIED


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from pathlib import Path

root = Path("/content/R5_work")

assert root.exists(), "R5 workspace not found. Restore the checkpoint first."

for p in sorted(root.rglob("*")):
    if p.is_file():
        print(p.relative_to(root))

R3_input/XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen1200_selection_agent_cv.csv
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen1200_test_split.csv
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen_oof_mass_policy.json
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_oof_feasible_mass_quantiles.csv
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_oof_training_calibration_rows.csv
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_seed42_model_reproduction.json
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_seed42_model_reproduction_cv.csv
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_training_calibration_rows.csv
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_training_feasible_mass_quantiles.csv
R3_recovered/XAgentic_R3_Candidate_Masking_FINAL/03_RESULTS_R

In [11]:
from google.colab import files

files.download(
    "/content/R5_work/R5_results/R5_FINAL_FINDINGS_AND_INTERPRETATION.md"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from google.colab import files
from pathlib import Path

zip_path = Path("/content/XAgentic_R5_FINAL_EVIDENCE.zip")

assert zip_path.exists(), "Final ZIP not found in this Colab session."

files.download(str(zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>